Here, we will try Understanding how to use Output Parsers..
Basically, an Output Parser is simply used to format the answer being given by the llm to us, in the form of json, or csv, or any type, depending on what format we want, and what type of outputparser() we use.
THe First 4 blocks here are just the setup required. The concept of output parser starts from the 5th Row.

In [1]:
%%capture
!pip install --force-reinstall --no-cache-dir tenacity==8.2.3 --user
!pip install "ibm-watsonx-ai==1.0.8" --user
!pip install "ibm-watson-machine-learning==1.0.367" --user
!pip install "langchain-ibm==0.1.7" --user
!pip install "langchain-community==0.2.10" --user
!pip install "langchain-experimental==0.0.62" --user
!pip install "langchainhub==0.1.18" --user
!pip install "langchain==0.2.11" --user
!pip install "pypdf==4.2.0" --user
!pip install "chromadb==0.4.24" --user

In [ ]:
import os
os._exit(00)

In [1]:
# You can also use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
import os
os.environ['ANONYMIZED_TELEMETRY'] = 'False'

from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM

By Now, I know the meaning of the box after this one, but just for the sake of revising, let me explain again.
1. So, as you can see, the variable 'model_id' contains the 'granite llm'.
2. The parameters are to configure the type of response we want from the llm. Like the max number of words it can use, and temperature for how much creativity.
3. Credentials is just the url to the ibm platform for the project.
4. The variable model just takes in the variables we have defined, and puts them in the variable of what our ibm platform needs
5. Now, since the granite model was a little weird, we tried using llama model, so for llama, you see how we created a new inference? llama_model.
6. We store the llm in watson

In [2]:
model_id = 'ibm/granite-4-h-small' 

parameters = {
    GenParams.MAX_NEW_TOKENS: 256,  # this controls the maximum number of tokens in the generated output
    GenParams.TEMPERATURE: 0.2, # this randomness or creativity of the model's responses 
}

credentials = {
    "url": "https://us-south.ml.cloud.ibm.com"
    # "api_key": "your api key here"
    # uncomment above and fill in the API key when running locally
}

project_id = "skills-network"

model = ModelInference(
    model_id=model_id,
    params=parameters,
    credentials=credentials,
    project_id=project_id
)
#Adding a llama model, because something is wrong with granite ig
llama_model = ModelInference(
    model_id='meta-llama/llama-4-maverick-17b-128e-instruct-fp8',
    params=parameters,
    credentials=credentials,
    project_id=project_id
)

Now, the coming 2 sentences are to create a LangChain interface for your IBM Granite model.
1. The first statement takes our first ModelInference configuration (model), which points to 'ibm/granite-4-h-small', and wraps it so LangChain can use it.
2. 2. The second statement creates a LangChain interface for our Meta Llama 4 Maverick model. It takes our second ModelInference configuration (llama_model), which points to 'meta-llama/llama-4-maverick...', and wraps it for LangChain.

In [7]:
granite_llm = WatsonxLLM(model = model)
llama_llm2 = WatsonxLLM(model=llama_model) #added llama model, because granite wasn't working ig

To use PromptTemplate, and the type of output parsers, we'll have to import it.

In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser

In [5]:
# Here, we create our JSON parser
json_parser = JsonOutputParser()

# Now, we create the format instructions manually. Not Necessary, sometimes, we can directly call it from the output_parser.get_format_instructions(), 
# but sometimes we can define it manually too.
format_instructions = """RESPONSE FORMAT: Return ONLY a single JSON object—no markdown, no examples, no extra keys.  It must look exactly like:
{
  "title": "movie title",
  "director": "director name",
  "year": 2000,
  "genre": "movie genre"
}

IMPORTANT: Your response must be *only* that JSON.  Do NOT include any illustrative or example JSON."""

# Create prompt template with instructions
prompt_template = PromptTemplate(
    template="""You are a JSON-only assistant.

Task: Generate info about the movie "{movie_name1}" in JSON format.

{format_instructions}
""",
    input_variables=["movie_name1"],
    partial_variables={"format_instructions": format_instructions},
)

# Create the chain
movie_chain =  prompt_template | llama_llm2 | json_parser

# Test with a movie name
movie_container = "The Matrix"
result = movie_chain.invoke({"movie_name1": movie_container})

# Print the structured result
print("Parsed result:")
print(f"Title: {result['title']}")
print(f"Director: {result['director']}")
print(f"Year: {result['year']}")
print(f"Genre: {result['genre']}")

Parsed result:
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction


In [6]:
movies = ["The Matrix", "Inception", "The Godfather"]

for movie in movies:
    result = movie_chain.invoke({"movie_name1": movie})
    print(f"\n=== {movie} ===")
    print(f"Title: {result['title']}")
    print(f"Director: {result['director']}")
    print(f"Year: {result['year']}")
    print(f"Genre: {result['genre']}")


=== The Matrix ===
Title: The Matrix
Director: The Wachowskis
Year: 1999
Genre: Science Fiction

=== Inception ===
Title: Inception
Director: Christopher Nolan
Year: 2010
Genre: Action, Sci-Fi

=== The Godfather ===
Title: The Godfather
Director: Francis Ford Coppola
Year: 1972
Genre: Crime, Drama


In [8]:
## Learning From this program:

Something I learnt here was: <br>
The question I had: <br>
See, in the examples before, we did: <br>
prompt = PromptTemplate( <br>
    template="Answer the user query. {format_instructions}\nList five {subject}.", <br>
    input_variables=["subject"],  # This variable will be provided when the chain is invoked <br>
    partial_variables={"format_instructions": format_instructions},  # This variable is set once when creating the prompt <br>
) <br>
But now, here we used: <br>
prompt_template = PromptTemplate( <br>
    template="""You are a JSON-only assistant. <br>
Task: Generate info about the movie "{movie_name1}" in JSON format. <br>
{format_instructions} <br>
""", <br>
    input_variables=["movie_name1"], <br>
    partial_variables={"format_instructions": format_instructions}, <br>
) <br>
....... <br>
So this much I understood that after writing the main template, we define input variables and parrtial variables, where we set format instructions. But then, what is the use of format instructions in the template itself? and can we literally just put it anywhere in the template like the first example? <br>
...............................................................................................................  <br>
Answer:  <br>
Yes we can put it anywhere in the template! It's just a placeholder like any other.  <br>
 <br>
Why put it in the template at all?  <br>
Because the LLM reads the entire template as one combined prompt. So {format_instructions} gets replaced with the actual instructions text and becomes part of the prompt the LLM sees.  <br>
Think of it like a letter template:  <br>
Dear {name},  <br>
Please send us {document_type}.  <br>
{format_instructions}  <br>
Regards  <br>

When filled in, the LLM sees:  <br>
Dear John,  <br>
Please send us your tax return.  <br>
Return ONLY a JSON object. No markdown. No extra text.  <br>
Regards  <br>

The instructions become part of the message the AI reads.  <br>
Yes, we can put it anywhere, but position matters for how the LLM prioritizes it:  <br>
At the beginning — AI sees formatting rules first, then the task:  <br>
{format_instructions}  <br>
Now do this: {query}  <br>

In the middle — task context first, then rules, then question:  <br>
You are a helpful assistant. <br>
{format_instructions}  <br>
Answer this: {query} <br>

At the end — AI reads the full task first, then formatting reminder: <br>
Answer this: {query} <br>
{format_instructions} <br>

Generally putting it before the actual query works best — the AI knows the rules before it starts generating. <br>